# LeetCode 1329 — Sort the Matrix Diagonally bằng Python

Link đề: https://leetcode.com/problems/sort-the-matrix-diagonally/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Matrix

Skeleton LeetCode:

```python
class Solution:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra **đặc trưng của một đường chéo** (hướng xuống-phải): mọi ô `(i, j)` trên cùng đường chéo có cùng hiệu `i − j`.
2. Biết liệt kê đủ `m + n − 1` đường chéo: bắt đầu từ **hàng đầu** và **cột đầu**.
3. Viết 5 cách: "nổi bọt" đổi chỗ kề chéo (baseline) → đi dọc từng đường chéo rồi `sort` → gom theo `i − j` bằng `dict` → heap cho từng đường chéo → counting sort (giá trị ≤ 100).
4. Phân tích độ phức tạp theo `m`, `n`, `k = min(m, n)` và biết khi nào counting sort thắng.
5. Tránh lỗi: dùng `i + j` (đường chéo ngược), chỉ đi từ hàng đầu, `pop()` sai chiều, sao chép nông ma trận.

## 2. Đề bài & ví dụ

Một **đường chéo ma trận** là dãy ô bắt đầu từ một ô ở **hàng trên cùng** hoặc **cột trái cùng**, đi theo hướng **xuống-phải** đến khi ra khỏi ma trận. Ví dụ với ma trận `6 × 3`, đường chéo bắt đầu ở `mat[2][0]` gồm `mat[2][0], mat[3][1], mat[4][2]`.

Cho ma trận `mat` cỡ `m × n`, hãy **sắp xếp tăng dần từng đường chéo** và trả về ma trận kết quả.

**Ví dụ 1:** `mat = [[3,3,1,1],[2,2,1,2],[1,1,1,2]]` → `[[1,1,1,1],[1,2,2,2],[1,2,3,3]]`

**Ví dụ 2:** `mat = [[11,25,66,1,69,7],[23,55,17,45,15,52],[75,31,36,44,58,8],[22,27,33,25,68,4],[84,28,14,11,5,50]]`
→ `[[5,17,4,1,52,7],[11,11,25,45,8,69],[14,23,25,44,58,15],[22,27,31,36,50,66],[84,28,75,33,55,68]]`

**Ràng buộc:** `1 <= m, n <= 100`; `1 <= mat[i][j] <= 100`.

In [1]:
import copy
import heapq
import random
import time
from collections import defaultdict
from typing import List

examples = [
    ([[3, 3, 1, 1], [2, 2, 1, 2], [1, 1, 1, 2]],
     [[1, 1, 1, 1], [1, 2, 2, 2], [1, 2, 3, 3]]),
    ([[11, 25, 66, 1, 69, 7], [23, 55, 17, 45, 15, 52], [75, 31, 36, 44, 58, 8], [22, 27, 33, 25, 68, 4], [84, 28, 14, 11, 5, 50]],
     [[5, 17, 4, 1, 52, 7], [11, 11, 25, 45, 8, 69], [14, 23, 25, 44, 58, 15], [22, 27, 31, 36, 50, 66], [84, 28, 75, 33, 55, 68]]),
]

def show(mat, title=""):
    if title:
        print(title)
    for row in mat:
        print("  " + " ".join(f"{v:3}" for v in row))

show(examples[0][0], "Ví dụ 1 — đầu vào:")
show(examples[0][1], "Ví dụ 1 — mong đợi:")

Ví dụ 1 — đầu vào:
    3   3   1   1
    2   2   1   2
    1   1   1   2
Ví dụ 1 — mong đợi:
    1   1   1   1
    1   2   2   2
    1   2   3   3


## 3. Kiến thức nền

### 3.1 Ma trận là list các list

`mat[i][j]` là hàng `i`, cột `j`; `m = len(mat)`, `n = len(mat[0])`. Duyệt mọi ô bằng hai vòng lặp lồng.

In [2]:
mat = [[1, 2, 3], [4, 5, 6]]
m, n = len(mat), len(mat[0])
print("m =", m, "n =", n)
for i in range(m):
    for j in range(n):
        print(f"({i},{j})={mat[i][j]}", end="  ")
    print()

m = 2 n = 3
(0,0)=1  (0,1)=2  (0,2)=3  
(1,0)=4  (1,1)=5  (1,2)=6  


### 3.2 Hiệu `i − j` xác định đường chéo

Đi xuống-phải: `(i, j) → (i+1, j+1)` ⇒ `i − j` **không đổi**. Do đó mỗi giá trị `d = i − j` ∈ `[−(n−1), m−1]` ứng với đúng một đường chéo ⇒ có `m + n − 1` đường chéo. (Ngược lại, `i + j` không đổi trên đường chéo **ngược** xuống-trái.)

In [3]:
m, n = 3, 4
for i in range(m):
    print("  ".join(f"{i - j:+d}" for j in range(n)), "   |   ", "  ".join(f"{i + j:2d}" for j in range(n)))
print("số đường chéo =", m + n - 1)

+0  -1  -2  -3    |     0   1   2   3
+1  +0  -1  -2    |     1   2   3   4
+2  +1  +0  -1    |     2   3   4   5
số đường chéo = 6


### 3.3 `defaultdict(list)` — gom nhóm

`defaultdict(list)` tự tạo list rỗng khi gặp khoá mới ⇒ `groups[key].append(x)` không cần kiểm tra khoá tồn tại.

In [4]:
groups = defaultdict(list)
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    groups[word[0]].append(word)
print(dict(groups))

{'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}


### 3.4 `sort`, `sorted`, `pop()` và `heapq`

- `lst.sort()` sắp tại chỗ; `sorted(x)` trả list mới; `reverse=True` để giảm dần.
- `lst.pop()` lấy phần tử **cuối** `O(1)`; `lst.pop(0)` lấy phần tử đầu nhưng `O(n)`.
- `heapq` là **min-heap**: `heappush`, `heappop` lấy phần tử nhỏ nhất `O(log k)`; `heapify` biến list thành heap `O(k)`.

In [5]:
a = [5, 1, 4, 2]
print(sorted(a), sorted(a, reverse=True))
b = sorted(a, reverse=True)
print("pop từ list giảm dần:", [b.pop() for _ in range(len(a))])
h = a[:]
heapq.heapify(h)
print("heappop lần lượt:", [heapq.heappop(h) for _ in range(len(a))])

[1, 2, 4, 5] [5, 4, 2, 1]
pop từ list giảm dần: [1, 2, 4, 5]
heappop lần lượt: [1, 2, 4, 5]


### 3.5 Sao chép ma trận: nông và sâu

`mat[:]` hoặc `list(mat)` chỉ sao chép **list ngoài**; các hàng vẫn dùng chung. Muốn bản sao độc lập: `[row[:] for row in mat]` hoặc `copy.deepcopy(mat)`.

In [6]:
orig = [[1, 2], [3, 4]]
shallow = orig[:]
deep = [row[:] for row in orig]
shallow[0][0] = 99
deep[1][1] = -1
print("orig:", orig, "| shallow:", shallow, "| deep:", deep)

orig: [[99, 2], [3, 4]] | shallow: [[99, 2], [3, 4]] | deep: [[1, 2], [3, -1]]


### 3.6 Counting sort cho giá trị nhỏ

Khi giá trị nằm trong khoảng nhỏ `[1, 100]`, đếm số lần xuất hiện `cnt[v]` rồi phát lại theo thứ tự tăng ⇒ sắp xếp `O(k + 100)` không cần so sánh.

In [7]:
vals = [5, 3, 5, 1, 3, 3]
cnt = [0] * 101
for v in vals:
    cnt[v] += 1
print([v for v in range(101) for _ in range(cnt[v])])

[1, 3, 3, 3, 5, 5]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (3 × 4): đánh dấu `d = i − j` cho từng ô:

```
d:  0 -1 -2 -3        mat:  3 3 1 1
    1  0 -1 -2              2 2 1 2
    2  1  0 -1              1 1 1 2
```

| d | các ô (theo thứ tự đi xuống) | giá trị | sau khi sort |
|---|---|---|---|
| −3 | (0,3) | 1 | 1 |
| −2 | (0,2),(1,3) | 1, 2 | 1, 2 |
| −1 | (0,1),(1,2),(2,3) | 3, 1, 2 | 1, 2, 3 |
| 0 | (0,0),(1,1),(2,2) | 3, 2, 1 | 1, 2, 3 |
| 1 | (1,0),(2,1) | 2, 1 | 1, 2 |
| 2 | (2,0) | 1 | 1 |

Ghi lại theo thứ tự đi xuống ⇒ `[[1,1,1,1],[1,2,2,2],[1,2,3,3]]` ✔.

In [8]:
mat = examples[0][0]
m, n = len(mat), len(mat[0])
for d in range(-(n - 1), m):
    cells = [(i, i - d) for i in range(m) if 0 <= i - d < n]
    vals = [mat[i][j] for i, j in cells]
    print(f"d={d:+d}: ô={cells} giá trị={vals} -> {sorted(vals)}")

d=-3: ô=[(0, 3)] giá trị=[1] -> [1]
d=-2: ô=[(0, 2), (1, 3)] giá trị=[1, 2] -> [1, 2]
d=-1: ô=[(0, 1), (1, 2), (2, 3)] giá trị=[3, 1, 2] -> [1, 2, 3]
d=+0: ô=[(0, 0), (1, 1), (2, 2)] giá trị=[3, 2, 1] -> [1, 2, 3]
d=+1: ô=[(1, 0), (2, 1)] giá trị=[2, 1] -> [1, 2]
d=+2: ô=[(2, 0)] giá trị=[1] -> [1]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** giống **bubble sort** nhưng theo đường chéo: lặp qua mọi ô, nếu `mat[i][j] > mat[i+1][j+1]` thì đổi chỗ. Lặp lại cả ma trận cho đến khi một lượt không có lần đổi nào. Mỗi lần đổi chỉ hoán vị hai phần tử **cùng đường chéo** nên đường chéo nào cũng dần được sắp tăng.

Ta làm trên **bản sao** để không sửa đầu vào.

In [9]:
def diagonal_sort_bubble(mat):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    changed = True
    while changed:
        changed = False
        for i in range(m - 1):
            for j in range(n - 1):
                if a[i][j] > a[i + 1][j + 1]:
                    a[i][j], a[i + 1][j + 1] = a[i + 1][j + 1], a[i][j]
                    changed = True
    return a

for mat, want in examples:
    got = diagonal_sort_bubble(mat)
    print("OK" if got == want else "SAI", got)

OK [[1, 1, 1, 1], [1, 2, 2, 2], [1, 2, 3, 3]]
OK [[5, 17, 4, 1, 52, 7], [11, 11, 25, 45, 8, 69], [14, 23, 25, 44, 58, 15], [22, 27, 31, 36, 50, 66], [84, 28, 75, 33, 55, 68]]


**Độ phức tạp:** mỗi lượt `O(m·n)`; đường chéo dài nhất có `k = min(m, n)` phần tử, bubble sort cần tối đa `k` lượt ⇒ `O(m·n·k)` — với `100 × 100` là `10^6` phép so sánh, chạy được nhưng chậm hơn nhiều. Bộ nhớ `O(1)` (ngoài bản sao).

**Nút thắt:** bubble sort bản chất `O(k²)` cho mỗi đường chéo. Nếu **tách riêng** mỗi đường chéo thành một list, ta dùng được `sort` `O(k log k)` của Python.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** các đường chéo **độc lập** với nhau (không ô nào thuộc hai đường chéo). Bài toán tách thành `m + n − 1` bài **sắp xếp mảng 1 chiều**.

Hai cách đánh số đường chéo:
1. **Theo ô bắt đầu:** `(0, j)` với `j = 0..n−1` và `(i, 0)` với `i = 1..m−1`; đi `(i+1, j+1)` đến khi ra ngoài.
2. **Theo khoá `d = i − j`:** duyệt mọi ô theo thứ tự hàng (row-major) — thứ tự này cũng là thứ tự **đi xuống** trên mỗi đường chéo.

**Quy tắc ghi lại:** nếu duyệt ô theo row-major thì ô gặp **trước** trên một đường chéo phải nhận giá trị **nhỏ hơn** ⇒ lấy dần từ nhỏ đến lớn (list giảm dần rồi `pop()`, hoặc heap `heappop`, hoặc con trỏ tăng dần).

**Invariant:** tập (multiset) giá trị trên mỗi đường chéo không đổi; chỉ thứ tự thay đổi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đi dọc từng đường chéo, `sort`, ghi lại

**Từng đoạn:**
1. Danh sách ô bắt đầu: `[(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]`.
2. Từ mỗi ô bắt đầu, đi `(i+k, j+k)` để thu các giá trị, `sort`.
3. Đi lại cùng đường đó, ghi các giá trị đã sắp.

In [10]:
def diagonal_sort_walk(mat):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    starts = [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]
    for si, sj in starts:
        vals = []
        i, j = si, sj
        while i < m and j < n:
            vals.append(a[i][j])
            i, j = i + 1, j + 1
        vals.sort()
        i, j = si, sj
        for v in vals:
            a[i][j] = v
            i, j = i + 1, j + 1
    return a

for mat, want in examples:
    print("OK" if diagonal_sort_walk(mat) == want else "SAI")

OK
OK


Code nộp LeetCode (Cách 2 — sửa tại chỗ, LeetCode chấp nhận):

In [11]:
class SolutionWalk:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        for si, sj in [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]:
            length = min(m - si, n - sj)
            vals = sorted(mat[si + k][sj + k] for k in range(length))
            for k in range(length):
                mat[si + k][sj + k] = vals[k]
        return mat

print([SolutionWalk().diagonalSort([r[:] for r in mat]) == want for mat, want in examples])

[True, True]


**Độ phức tạp:** tổng độ dài các đường chéo là `m·n`; mỗi đường chéo dài `≤ k` ⇒ `O(m·n·log k)` thời gian, `O(k)` bộ nhớ phụ.

### 7.2 Cách 3 — Gom theo khoá `i − j` bằng `dict`

**Từng đoạn:**
1. Lượt 1 (row-major): `groups[i − j].append(mat[i][j])`.
2. Sắp mỗi nhóm **giảm dần** để `pop()` (cuối list, `O(1)`) trả phần tử **nhỏ nhất**.
3. Lượt 2 (row-major): `mat[i][j] = groups[i − j].pop()`.

In [12]:
def diagonal_sort_dict(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    for i in range(m):
        for j in range(n):
            groups[i - j].append(mat[i][j])
    for d in groups:
        groups[d].sort(reverse=True)
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            res[i][j] = groups[i - j].pop()
    return res

for mat, want in examples:
    print("OK" if diagonal_sort_dict(mat) == want else "SAI")

OK
OK


Code nộp LeetCode (Cách 3):

In [13]:
class SolutionDict:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        groups = defaultdict(list)
        for i in range(m):
            for j in range(n):
                groups[i - j].append(mat[i][j])
        for vals in groups.values():
            vals.sort(reverse=True)
        for i in range(m):
            for j in range(n):
                mat[i][j] = groups[i - j].pop()
        return mat

print([SolutionDict().diagonalSort([r[:] for r in mat]) == want for mat, want in examples])

[True, True]


**Độ phức tạp:** `O(m·n·log k)` thời gian, `O(m·n)` bộ nhớ (lưu mọi giá trị trong `groups`).

### 7.3 Cách 4 — Một min-heap cho mỗi đường chéo

Thay vì sort rồi `pop()`, đẩy giá trị vào **min-heap** của đường chéo; khi ghi lại theo row-major, `heappop` luôn trả phần tử nhỏ nhất còn lại. Kết quả giống hệt, nhưng minh hoạ một công cụ hữu ích khi dữ liệu đến dần (streaming).

In [14]:
def diagonal_sort_heap(mat):
    m, n = len(mat), len(mat[0])
    heaps = defaultdict(list)
    for i in range(m):
        for j in range(n):
            heapq.heappush(heaps[i - j], mat[i][j])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            res[i][j] = heapq.heappop(heaps[i - j])
    return res

for mat, want in examples:
    print("OK" if diagonal_sort_heap(mat) == want else "SAI")

OK
OK


Code nộp LeetCode (Cách 4):

In [15]:
class SolutionHeap:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        heaps = defaultdict(list)
        for i in range(m):
            for j in range(n):
                heapq.heappush(heaps[i - j], mat[i][j])
        for i in range(m):
            for j in range(n):
                mat[i][j] = heapq.heappop(heaps[i - j])
        return mat

print([SolutionHeap().diagonalSort([r[:] for r in mat]) == want for mat, want in examples])

[True, True]


**Độ phức tạp:** mỗi push/pop `O(log k)` ⇒ `O(m·n·log k)` thời gian, `O(m·n)` bộ nhớ.

### 7.4 Cách 5 — Counting sort trên từng đường chéo (giá trị `1..100`)

Vì `1 <= mat[i][j] <= 100`, với mỗi đường chéo ta đếm `cnt[v]` (101 ô), rồi đi lại đường chéo, ghi lần lượt các giá trị từ nhỏ đến lớn theo `cnt`. Mỗi đường chéo tốn `O(len + 100)` ⇒ tổng `O(m·n + 100·(m + n))` — **tuyến tính** theo kích thước ma trận.

In [16]:
def diagonal_sort_counting(mat, max_val=100):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    for si, sj in [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]:
        cnt = [0] * (max_val + 1)
        i, j = si, sj
        while i < m and j < n:
            cnt[a[i][j]] += 1
            i, j = i + 1, j + 1
        i, j, v = si, sj, 0
        while i < m and j < n:
            while cnt[v] == 0:
                v += 1
            a[i][j] = v
            cnt[v] -= 1
            i, j = i + 1, j + 1
    return a

for mat, want in examples:
    print("OK" if diagonal_sort_counting(mat) == want else "SAI")

OK
OK


Code nộp LeetCode (Cách 5):

In [17]:
class SolutionCounting:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        for si, sj in [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]:
            length = min(m - si, n - sj)
            cnt = [0] * 101
            for k in range(length):
                cnt[mat[si + k][sj + k]] += 1
            v = 0
            for k in range(length):
                while cnt[v] == 0:
                    v += 1
                mat[si + k][sj + k] = v
                cnt[v] -= 1
        return mat

print([SolutionCounting().diagonalSort([r[:] for r in mat]) == want for mat, want in examples])

[True, True]


**Độ phức tạp:** `O(m·n + 101·(m + n − 1))` thời gian, `O(101)` bộ nhớ phụ. Trên Python, `sort` (viết bằng C) thường vẫn nhanh ngang hoặc hơn với kích thước nhỏ, nhưng về lý thuyết đây là cách tối ưu khi miền giá trị nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1**: lượt gom, sắp giảm dần, rồi lượt ghi lại (in `groups` sau mỗi lần `pop`).

| lượt ghi (i,j) | d | groups[d] trước pop | giá trị ghi |
|---|---|---|---|
| (0,0) | 0 | [3,2,1] | 1 |
| (0,1) | −1 | [3,2,1] | 1 |
| (0,2) | −2 | [2,1] | 1 |
| (0,3) | −3 | [1] | 1 |
| (1,0) | 1 | [2,1] | 1 |
| (1,1) | 0 | [3,2] | 2 |
| … | … | … | … |

In [18]:
def trace_dict(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    for i in range(m):
        for j in range(n):
            groups[i - j].append(mat[i][j])
    print("sau khi gom :", dict(sorted(groups.items())))
    for d in groups:
        groups[d].sort(reverse=True)
    print("sắp giảm dần:", dict(sorted(groups.items())))
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            before = groups[i - j][:]
            res[i][j] = groups[i - j].pop()
            print(f"  ({i},{j}) d={i - j:+d}: {before} -> ghi {res[i][j]}")
    show(res, "kết quả:")
    return res

trace_dict(examples[0][0])

sau khi gom : {-3: [1], -2: [1, 2], -1: [3, 1, 2], 0: [3, 2, 1], 1: [2, 1], 2: [1]}
sắp giảm dần: {-3: [1], -2: [2, 1], -1: [3, 2, 1], 0: [3, 2, 1], 1: [2, 1], 2: [1]}
  (0,0) d=+0: [3, 2, 1] -> ghi 1
  (0,1) d=-1: [3, 2, 1] -> ghi 1
  (0,2) d=-2: [2, 1] -> ghi 1
  (0,3) d=-3: [1] -> ghi 1
  (1,0) d=+1: [2, 1] -> ghi 1
  (1,1) d=+0: [3, 2] -> ghi 2
  (1,2) d=-1: [3, 2] -> ghi 2
  (1,3) d=-2: [2] -> ghi 2
  (2,0) d=+2: [1] -> ghi 1
  (2,1) d=+1: [2] -> ghi 2
  (2,2) d=+0: [3] -> ghi 3
  (2,3) d=-1: [3] -> ghi 3
kết quả:
    1   1   1   1
    1   2   2   2
    1   2   3   3


[[1, 1, 1, 1], [1, 2, 2, 2], [1, 2, 3, 3]]

Trace Cách 1 (bubble) — số lượt và số lần đổi chỗ trên Ví dụ 2:

In [19]:
def trace_bubble(mat):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    rnd = 0
    while True:
        swaps = 0
        for i in range(m - 1):
            for j in range(n - 1):
                if a[i][j] > a[i + 1][j + 1]:
                    a[i][j], a[i + 1][j + 1] = a[i + 1][j + 1], a[i][j]
                    swaps += 1
        rnd += 1
        print(f"lượt {rnd}: {swaps} lần đổi chỗ")
        if swaps == 0:
            break
    return a

print(trace_bubble(examples[1][0]) == examples[1][1])

lượt 1: 13 lần đổi chỗ
lượt 2: 5 lần đổi chỗ
lượt 3: 3 lần đổi chỗ
lượt 4: 1 lần đổi chỗ
lượt 5: 0 lần đổi chỗ
True


## 9. Code nộp LeetCode

Bản chọn nộp: gom theo `i − j` + sort giảm dần + `pop()` (Cách 3) — ngắn, rõ, `O(m·n·log k)`.

In [20]:
class Solution:
    def diagonalSort(self, mat: list[list[int]]) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        diag = defaultdict(list)
        for i in range(m):
            for j in range(n):
                diag[i - j].append(mat[i][j])
        for vals in diag.values():
            vals.sort(reverse=True)            # pop() sẽ trả phần tử nhỏ nhất
        for i in range(m):
            for j in range(n):
                mat[i][j] = diag[i - j].pop()
        return mat

for mat, want in examples:
    got = Solution().diagonalSort([r[:] for r in mat])
    print("OK" if got == want else "SAI")

OK
OK


## 10. Vì sao thuật toán đúng

1. **Phân hoạch đúng:** ô `(i, j)` và `(i', j')` cùng đường chéo ⇔ tồn tại `t` với `i' = i + t`, `j' = j + t` ⇔ `i − j = i' − j'`. Vậy khoá `i − j` gom **đúng và đủ** các ô của từng đường chéo; các đường chéo không giao nhau ⇒ sắp xếp riêng từng nhóm không ảnh hưởng nhau.
2. **Thứ tự ghi đúng:** trên một đường chéo, ô `(i, j)` đứng trước `(i+1, j+1)` cả theo hướng đi xuống lẫn theo thứ tự row-major (hàng nhỏ hơn được duyệt trước). Ở lượt ghi, mỗi lần gặp một ô của đường chéo `d` ta lấy phần tử **nhỏ nhất còn lại** của nhóm `d` ⇒ các ô của `d` nhận giá trị **không giảm** theo hướng đi xuống — tức đường chéo đã được sắp tăng.
3. **Bảo toàn giá trị:** mỗi giá trị được lấy ra đúng một lần (số phần tử nhóm = số ô của đường chéo) ⇒ multiset không đổi.

Với bubble sort: mỗi lần đổi giảm số **nghịch thế** trên một đường chéo đi 1; khi không còn cặp kề nào nghịch thế thì mọi đường chéo đã tăng dần.

## 11. Phân tích Time / Space

Đặt `m, n` là kích thước, `k = min(m, n)` (độ dài đường chéo dài nhất), `V = 100` (giá trị lớn nhất).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Bubble chéo | `O(m·n·k)` | `O(1)` |
| 2. Đi dọc + sort | `O(m·n·log k)` | `O(k)` |
| 3. Dict `i − j` + sort | `O(m·n·log k)` | `O(m·n)` |
| 4. Heap mỗi đường chéo | `O(m·n·log k)` | `O(m·n)` |
| 5. Counting sort | `O(m·n + V·(m + n))` | `O(V)` |

Với `m = n = 100`: `m·n = 10^4`, mọi cách trừ bubble đều chạy trong vài mili-giây.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Các bản nộp sửa ma trận **tại chỗ** ⇒ luôn truyền **bản sao sâu**.

In [21]:
solvers = {
    "bubble": diagonal_sort_bubble,
    "walk": diagonal_sort_walk,
    "dict": diagonal_sort_dict,
    "heap": diagonal_sort_heap,
    "counting": diagonal_sort_counting,
    "submit": lambda a: Solution().diagonalSort(a),
    "submit_walk": lambda a: SolutionWalk().diagonalSort(a),
    "submit_dict": lambda a: SolutionDict().diagonalSort(a),
    "submit_heap": lambda a: SolutionHeap().diagonalSort(a),
    "submit_count": lambda a: SolutionCounting().diagonalSort(a),
}
cases = examples + [
    ([[7]], [[7]]),                                   # 1 x 1
    ([[3, 1, 2]], [[3, 1, 2]]),                       # 1 hàng: mỗi đường chéo 1 ô
    ([[3], [1], [2]], [[3], [1], [2]]),               # 1 cột
    ([[9, 8, 7], [6, 5, 4], [3, 2, 1]], [[1, 4, 7], [2, 5, 8], [3, 6, 9]]),
    ([[5, 5], [5, 5]], [[5, 5], [5, 5]]),             # toàn bằng nhau
    ([[2, 1], [1, 1]], [[1, 1], [1, 2]]),
]
for mat, want in cases:
    for name, f in solvers.items():
        assert f(copy.deepcopy(mat)) == want, (name, mat)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 8 case qua cho 10 lời giải


### 12.2 Hàm kiểm tra tính hợp lệ + ngẫu nhiên (seed cố định)

`is_valid(orig, res)` kiểm tra độc lập: mỗi đường chéo của `res` tăng dần **và** có cùng multiset với đường chéo tương ứng của `orig`.

In [22]:
def diagonals(mat):
    m, n = len(mat), len(mat[0])
    return {d: [mat[i][i - d] for i in range(m) if 0 <= i - d < n] for d in range(-(n - 1), m)}

def is_valid(orig, res):
    if len(orig) != len(res) or any(len(a) != len(b) for a, b in zip(orig, res)):
        return False
    do, dr = diagonals(orig), diagonals(res)
    return all(dr[d] == sorted(do[d]) for d in do)

rng = random.Random(1329)
for _ in range(500):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    hi = rng.choice([3, 10, 100])
    mat = [[rng.randint(1, hi) for _ in range(n)] for _ in range(m)]
    snapshot = copy.deepcopy(mat)
    want = diagonal_sort_bubble(mat)
    assert mat == snapshot                    # bubble không sửa đầu vào
    assert is_valid(mat, want)
    for name, f in solvers.items():
        assert f(copy.deepcopy(mat)) == want, (name, mat)
print("500 ma trận ngẫu nhiên: mọi cách khớp và hợp lệ")

500 ma trận ngẫu nhiên: mọi cách khớp và hợp lệ


### 12.3 Kích thước lớn nhất `100 × 100`

In [23]:
rng = random.Random(100)
big = [[rng.randint(1, 100) for _ in range(100)] for _ in range(100)]
ref = None
for name in ["walk", "dict", "heap", "counting", "submit", "bubble"]:
    t0 = time.time()
    out = solvers[name](copy.deepcopy(big))
    print(f"{name:9}: {(time.time() - t0) * 1000:7.1f} ms")
    ref = ref or out
    assert out == ref
assert is_valid(big, ref)

walk     :     3.5 ms
dict     :     3.6 ms
heap     :     4.1 ms


counting :     4.4 ms
submit   :     3.5 ms


bubble   :    58.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng khoá `i + j` (đường chéo ngược)

In [24]:
def wrong_anti_key(mat):
    m, n = len(mat), len(mat[0])
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            g[i + j].append(mat[i][j])                # SAI: i + j là đường chéo ngược
    for v in g.values():
        v.sort(reverse=True)
    return [[g[i + j].pop() for j in range(n)] for i in range(m)]

mat, want = examples[0]
show(wrong_anti_key(copy.deepcopy(mat)), "sai:")
show(want, "đúng:")

sai:
    3   2   1   1
    3   1   1   1
    2   1   2   2
đúng:
    1   1   1   1
    1   2   2   2
    1   2   3   3


**Vì sao sai:** `i + j` không đổi trên hướng **xuống-trái**, nên ta đã sắp các đường chéo ngược. **Sửa:** khoá `i − j`.

### Lỗi 2 — Chỉ đi từ các ô ở hàng đầu

In [25]:
def wrong_top_only(mat):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    for sj in range(n):                                # SAI: thiếu các ô bắt đầu (i, 0), i >= 1
        length = min(m, n - sj)
        vals = sorted(a[k][sj + k] for k in range(length))
        for k in range(length):
            a[k][sj + k] = vals[k]
    return a

mat, want = examples[1]
got = wrong_top_only(mat)
print("đúng không?", got == want)
print("hàng cuối sai :", got[-1], "\nhàng cuối đúng:", want[-1])

đúng không? False
hàng cuối sai : [84, 28, 14, 11, 55, 68] 
hàng cuối đúng: [84, 28, 75, 33, 55, 68]


**Vì sao sai:** các đường chéo bắt đầu ở cột 0 (từ hàng 1 trở xuống) không bao giờ được sắp. **Sửa:** ô bắt đầu gồm `(0, j)` **và** `(i, 0)` với `i ≥ 1`.

### Lỗi 3 — Sắp tăng dần rồi `pop()`

In [26]:
def wrong_pop_order(mat):
    m, n = len(mat), len(mat[0])
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            g[i - j].append(mat[i][j])
    for v in g.values():
        v.sort()                                       # SAI: tăng dần thì pop() trả phần tử lớn nhất
    return [[g[i - j].pop() for j in range(n)] for i in range(m)]

mat, want = examples[0]
show(wrong_pop_order(copy.deepcopy(mat)), "sai (mỗi đường chéo bị sắp GIẢM):")

sai (mỗi đường chéo bị sắp GIẢM):
    3   3   2   1
    2   2   2   1
    1   1   1   1


**Vì sao sai:** `pop()` lấy phần tử **cuối**. Với list tăng dần, đó là phần tử lớn nhất ⇒ mỗi đường chéo thành giảm dần. **Sửa:** `sort(reverse=True)` rồi `pop()`, hoặc giữ tăng dần và dùng con trỏ/`deque.popleft()`.

### Lỗi 4 — Sao chép nông rồi tưởng đầu vào còn nguyên

In [27]:
def solve_shallow(mat):
    a = mat[:]                                         # SAI: các hàng vẫn dùng chung với mat
    m, n = len(a), len(a[0])
    for si, sj in [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]:
        length = min(m - si, n - sj)
        vals = sorted(a[si + k][sj + k] for k in range(length))
        for k in range(length):
            a[si + k][sj + k] = vals[k]
    return a

orig = [[3, 3, 1, 1], [2, 2, 1, 2], [1, 1, 1, 2]]
before = copy.deepcopy(orig)
res = solve_shallow(orig)
print("đầu vào bị thay đổi?", orig != before)
do_nothing = lambda mat: [row[:] for row in mat]          # "lời giải" không làm gì cả
print("lời giải 'không làm gì' chạy SAU trên cùng orig có khớp đáp án không?",
      do_nothing(orig) == examples[0][1], "<- test bị đánh lừa")
print("chạy trên bản gốc thật:", do_nothing(before) == examples[0][1])

đầu vào bị thay đổi? True
lời giải 'không làm gì' chạy SAU trên cùng orig có khớp đáp án không? True <- test bị đánh lừa
chạy trên bản gốc thật: False


**Vì sao sai:** `mat[:]` chỉ sao chép list ngoài; sửa `a[i][j]` là sửa luôn `mat[i][j]`. Kết quả vẫn đúng, nhưng đầu vào bị phá — khi test đối chiếu nhiều lời giải trên **cùng** một ma trận, các lời giải sau nhận đầu vào đã sắp sẵn, che giấu lỗi. **Sửa:** `[row[:] for row in mat]` hoặc `copy.deepcopy`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "**đường chéo**" ⇒ nghĩ ngay khoá `i − j` (xuống-phải) hoặc `i + j` (xuống-trái).
- "Xử lý từng nhóm độc lập rồi ghi lại đúng vị trí" ⇒ **gom bằng dict** + duyệt lại theo **cùng thứ tự**; dùng list giảm dần + `pop()` để lấy nhỏ nhất `O(1)`.
- Miền giá trị nhỏ (≤ 100, ≤ 1000) ⇒ cân nhắc **counting sort**.
- Liệt kê đường chéo: ô bắt đầu là **hàng đầu + cột đầu** (`m + n − 1` ô).
- Test ma trận: luôn truyền **bản sao sâu**, viết `is_valid` kiểm tra tính chất.

## 15. Biến thể của bài

1. **Sắp giảm dần** từng đường chéo: đổi chiều sort.
2. **LeetCode 3446 — Sort Matrix by Diagonals:** nửa dưới (gồm chéo chính) giảm dần, nửa trên tăng dần — chỉ khác chiều sort theo dấu của `d`.
3. **LeetCode 498 — Diagonal Traverse:** duyệt zigzag theo đường chéo ngược `i + j`.
4. **LeetCode 766 — Toeplitz Matrix:** kiểm tra mọi đường chéo có giá trị bằng nhau: `mat[i][j] == mat[i-1][j-1]`.
5. **Sắp đường chéo ngược** (`i + j`) — Bài tập 5.

In [28]:
def is_toeplitz(mat):
    """LeetCode 766."""
    return all(mat[i][j] == mat[i - 1][j - 1] for i in range(1, len(mat)) for j in range(1, len(mat[0])))

def sort_by_diagonals_3446(grid):
    """LeetCode 3446: d >= 0 (nửa dưới) giảm dần, d < 0 (nửa trên) tăng dần."""
    n, m = len(grid), len(grid[0])
    g = defaultdict(list)
    for i in range(n):
        for j in range(m):
            g[i - j].append(grid[i][j])
    for d, v in g.items():
        v.sort(reverse=(d < 0))            # pop() lấy cuối: d<0 cần nhỏ trước, d>=0 cần lớn trước
    return [[g[i - j].pop() for j in range(m)] for i in range(n)]

print(is_toeplitz([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]]), is_toeplitz([[1, 2], [2, 2]]))
print(sort_by_diagonals_3446([[1, 7, 3], [9, 8, 2], [4, 5, 6]]))

True False
[[8, 2, 3], [9, 6, 7], [4, 5, 1]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bubble chéo | đổi chỗ cặp kề chéo đến khi ổn định | `O(m·n·k)` | `O(1)` | kiểm chứng, khi cấm bộ nhớ phụ |
| 2. Đi dọc + sort | từ mỗi ô bắt đầu, thu – sort – ghi | `O(m·n·log k)` | `O(k)` | ít bộ nhớ, dễ hiểu |
| 3. Dict `i − j` | gom row-major, sort giảm, `pop()` | `O(m·n·log k)` | `O(m·n)` | ngắn gọn nhất, lời giải nộp |
| 4. Heap | min-heap mỗi đường chéo | `O(m·n·log k)` | `O(m·n)` | dữ liệu đến dần |
| 5. Counting sort | đếm giá trị `1..100` mỗi đường chéo | `O(m·n + V(m+n))` | `O(V)` | miền giá trị nhỏ, cần tuyến tính |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().diagonalSort([[9,8,7],[6,5,4],[3,2,1]])` trả về gì?

### Bài 2 (tính tay)
Ma trận `3 × 5` có bao nhiêu đường chéo? Liệt kê độ dài từng đường chéo theo `d = −4..2`.

### Bài 3 (điền TODO)
Hoàn thiện `diag_sort_todo` theo Cách 3.

### Bài 4 (tự viết lại)
Viết lại Cách 5 (counting sort) dùng khoá `i − j` thay vì ô bắt đầu: `diag_sort_counting_ex(mat)`.

### Bài 5 (mini challenge)
Viết `anti_diagonal_sort(mat)`: sắp tăng dần mọi **đường chéo ngược** (hướng xuống-trái, khoá `i + j`), thứ tự tính theo hàng tăng dần. Kiểm chứng bằng một bản bubble "đổi chỗ `(i, j)` với `(i+1, j−1)`".

In [29]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def diag_sort_todo(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    # TODO: gom theo i - j
    # TODO: sort mỗi nhóm giảm dần
    # TODO: ghi lại bằng pop()
    return mat

def diag_sort_counting_ex_todo(mat):
    # TODO
    pass

def anti_diagonal_sort_todo(mat):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Đường chéo chính `9,5,1`. (2) Sắp ⇒ `1,5,9`; các đường chéo phụ `8,4 → 4,8`, `6,2 → 2,6`. (3) Ghép lại thành ma trận "tăng theo cột".

**Bài 2:** (1) `m + n − 1 = 7`. (2) Độ dài = số `i ∈ [0, m)` sao cho `0 ≤ i − d < n`. (3) `d = −4..2` ⇒ `1, 2, 3, 3, 3, 2, 1`.

**Bài 3:** (1) `groups[i - j].append(mat[i][j])`. (2) `for v in groups.values(): v.sort(reverse=True)`. (3) `mat[i][j] = groups[i - j].pop()`.

**Bài 4:** (1) `cnt = defaultdict(lambda: [0] * 101)`. (2) Lượt 1 tăng `cnt[i - j][v]`. (3) Lượt 2 giữ con trỏ `ptr[d]` tăng dần tới giá trị còn đếm > 0.

**Bài 5:** (1) Khoá `i + j`. (2) Duyệt row-major: ô có hàng nhỏ hơn trên cùng đường chéo ngược được gặp trước. (3) Sort giảm dần + `pop()`; bubble kiểm chứng so sánh `a[i][j] > a[i+1][j-1]`.

## 19. Đáp án tham khảo

In [30]:
# Bài 1
b1 = Solution().diagonalSort([[9, 8, 7], [6, 5, 4], [3, 2, 1]])
print("Bài 1:", b1)
assert b1 == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]

# Bài 2
lengths = [len(v) for d, v in sorted(diagonals([[0] * 5 for _ in range(3)]).items())]
print("Bài 2:", len(lengths), "đường chéo, độ dài", lengths)
assert lengths == [1, 2, 3, 3, 3, 2, 1]

# Bài 3
def diag_sort_sol(mat):
    m, n = len(mat), len(mat[0])
    groups = defaultdict(list)
    for i in range(m):
        for j in range(n):
            groups[i - j].append(mat[i][j])
    for v in groups.values():
        v.sort(reverse=True)
    for i in range(m):
        for j in range(n):
            mat[i][j] = groups[i - j].pop()
    return mat

# Bài 4
def diag_sort_counting_ex(mat):
    m, n = len(mat), len(mat[0])
    cnt = defaultdict(lambda: [0] * 101)
    for i in range(m):
        for j in range(n):
            cnt[i - j][mat[i][j]] += 1
    ptr = defaultdict(int)
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            d = i - j
            while cnt[d][ptr[d]] == 0:
                ptr[d] += 1
            res[i][j] = ptr[d]
            cnt[d][ptr[d]] -= 1
    return res

rng = random.Random(3)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    mat = [[rng.randint(1, 100) for _ in range(n)] for _ in range(m)]
    want = diagonal_sort_bubble(mat)
    assert diag_sort_sol(copy.deepcopy(mat)) == want
    assert diag_sort_counting_ex(mat) == want
print("Bài 1–4: qua")

Bài 1: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Bài 2: 7 đường chéo, độ dài [1, 2, 3, 3, 3, 2, 1]
Bài 1–4: qua


In [31]:
# Bài 5
def anti_diagonal_sort(mat):
    m, n = len(mat), len(mat[0])
    g = defaultdict(list)
    for i in range(m):
        for j in range(n):
            g[i + j].append(mat[i][j])
    for v in g.values():
        v.sort(reverse=True)
    return [[g[i + j].pop() for j in range(n)] for i in range(m)]

def anti_diagonal_bubble(mat):
    a = [row[:] for row in mat]
    m, n = len(a), len(a[0])
    changed = True
    while changed:
        changed = False
        for i in range(m - 1):
            for j in range(1, n):
                if a[i][j] > a[i + 1][j - 1]:
                    a[i][j], a[i + 1][j - 1] = a[i + 1][j - 1], a[i][j]
                    changed = True
    return a

print("Bài 5:", anti_diagonal_sort([[9, 8, 7], [6, 5, 4], [3, 2, 1]]))
rng = random.Random(5)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    mat = [[rng.randint(1, 20) for _ in range(n)] for _ in range(m)]
    assert anti_diagonal_sort(mat) == anti_diagonal_bubble(mat)
print("Bài 5: 300 ma trận ngẫu nhiên khớp bubble chéo ngược")

Bài 5: [[9, 6, 3], [8, 5, 2], [7, 4, 1]]
Bài 5: 300 ma trận ngẫu nhiên khớp bubble chéo ngược


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template gom theo đường chéo:**

```python
from collections import defaultdict
g = defaultdict(list)
for i in range(m):
    for j in range(n):
        g[i - j].append(mat[i][j])        # i + j cho đường chéo ngược
for v in g.values():
    v.sort(reverse=True)                  # pop() -> nhỏ nhất
for i in range(m):
    for j in range(n):
        mat[i][j] = g[i - j].pop()
```

**Ô bắt đầu của mọi đường chéo:** `[(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]`.

**Checklist trước khi submit:**
- [ ] Khoá `i − j` (không phải `i + j`).
- [ ] Đủ `m + n − 1` đường chéo (hàng đầu **và** cột đầu).
- [ ] Chiều sort khớp với cách lấy (`reverse=True` + `pop()`).
- [ ] Thử ma trận `1 × n`, `m × 1`, `1 × 1`.
- [ ] Không sao chép nông khi cần giữ đầu vào.
- [ ] Đúng chữ ký: `diagonalSort(self, mat: list[list[int]]) -> list[list[int]]`.